# 00 — Configuração do ambiente
## Análise Exploratória de Dados sobre a Incidência de Dengue em Santa Catarina

Este notebook apresenta uma Análise Exploratória de Dados (EDA) sobre
a incidência de dengue nos municípios do estado de Santa Catarina.

A análise considera inicialmente o período de 2000 a 2025, utilizando
dados públicos disponibilizados pelo Ministério da Saúde por meio do
Sistema de Informação de Agravos de Notificação (SINAN).

Posteriormente, os dados epidemiológicos serão relacionados com
informações climáticas, demográficas e socioeconômicas provenientes
de bases públicas brasileiras, buscando investigar fatores
possivelmente associados à incidência de dengue.


In [6]:
%pip install requests pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 4.2 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 1.6 MB/s  0:00:01 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 2.5 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [matplotlib]7 [matplotlib]
Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import requests
import time

from datetime import datetime

Matplotlib is building the font cache; this may take a moment.


In [8]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# 1. Introdução

A dengue é uma doença viral transmitida principalmente pela picada
da fêmea infectada do mosquito *Aedes aegypti*.

A ocorrência da doença não depende exclusivamente da presença do
vetor, podendo apresentar relações com diferentes características
ambientais, climáticas, demográficas e socioeconômicas.

Dessa forma, este trabalho propõe a realização de uma análise
exploratória dos registros de dengue em Santa Catarina, buscando
identificar padrões espaciais e temporais da doença e investigar
possíveis associações entre sua incidência e diferentes
características dos municípios.

A análise exploratória constitui uma etapa inicial do projeto e
servirá como base para a seleção de variáveis que poderão ser
posteriormente utilizadas na construção de modelos de Machine
Learning.

## 2.1 Objetivo geral

Analisar a incidência de dengue nos municípios de Santa Catarina
entre 2000 e 2025, buscando identificar padrões temporais e espaciais
e investigar possíveis associações com fatores climáticos,
demográficos e socioeconômicos.

## 2.2 Objetivos específicos

- Analisar a evolução histórica dos registros de dengue em Santa Catarina;

- identificar padrões sazonais na ocorrência da doença;

- comparar a incidência de dengue entre os municípios catarinenses;

- investigar possíveis relações entre temperatura, precipitação,
  umidade e incidência de dengue;

- investigar possíveis relações entre características demográficas
  e socioeconômicas dos municípios e a incidência da doença;

- avaliar a influência de informações temporais anteriores,
  como casos e condições climáticas dos meses anteriores;

- identificar variáveis candidatas para utilização posterior em
  modelos de Machine Learning.

# 3. Hipóteses de pesquisa

A partir das características biológicas do *Aedes aegypti* e dos
fatores potencialmente relacionados à transmissão da dengue,
serão investigadas as seguintes hipóteses:

**H1 — Sazonalidade**

A incidência de dengue apresenta comportamento sazonal ao longo
dos meses do ano.

**H2 — Temperatura**

A temperatura apresenta associação com a incidência de dengue.

**H3 — Precipitação**

A precipitação observada em períodos anteriores apresenta
associação com a incidência posterior de dengue.

**H4 — Umidade**

A umidade relativa do ar apresenta associação com a incidência
da doença.

**H5 — Densidade demográfica**

Municípios com diferentes níveis de densidade demográfica
apresentam padrões distintos de incidência de dengue.

**H6 — Condições socioeconômicas**

Características socioeconômicas e de infraestrutura urbana
apresentam associação com diferenças na incidência entre
municípios.

**H7 — Dependência temporal**

A incidência observada nos meses anteriores apresenta relação
com a incidência observada no mês atual.

# 4. Fontes dos dados

## 4.1 Sistema de Informação de Agravos de Notificação — SINAN

Os registros epidemiológicos de dengue utilizados neste estudo
serão obtidos a partir dos dados públicos disponibilizados pelo
Ministério da Saúde.

O Ministério da Saúde disponibiliza uma API de Dados Abertos com
endpoint específico para consulta de registros de dengue.

Os microdados serão utilizados inicialmente para construir uma
base agregada com granularidade:

**município de residência × mês**

O período considerado será de 2000 a 2025 e o recorte geográfico
principal será o estado de Santa Catarina.

In [3]:
import requests
import pandas as pd
import json

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 1.3 MB/s  0:00:07eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 2.2 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [9]:
URL_DENGUE = "https://apidadosabertos.saude.gov.br/arboviroses/dengue"

parametros = {
    "nu_ano": "2025",
    "limit": 5,
    "offset": 0
}

resposta = requests.get(
    URL_DENGUE,
    params=parametros,
    timeout=60
)

print("Status:", resposta.status_code)
print("URL:", resposta.url)
print("Content-Type:", resposta.headers.get("content-type"))

Status: 200
URL: https://apidadosabertos.saude.gov.br/arboviroses/dengue?nu_ano=2025&limit=5&offset=0
Content-Type: application/json


In [10]:
dados = resposta.json()

print("Tipo retornado:", type(dados))

Tipo retornado: <class 'dict'>


In [11]:
if isinstance(dados, dict):
    print("Chaves disponíveis:")
    print(dados.keys())
else:
    print("Quantidade de itens:", len(dados))

Chaves disponíveis:
dict_keys(['dengue'])


In [12]:
print(
    json.dumps(
        dados,
        indent=2,
        ensure_ascii=False
    )[:5000]
)

{
  "dengue": [
    {
      "tp_not": "2",
      "id_agravo": "A90",
      "dt_notific": "2025-03-11",
      "sem_not": "202511",
      "nu_ano": "2025",
      "sg_uf_not": "35",
      "id_municip": "354990",
      "id_regiona": "1351",
      "id_unidade": "3708616",
      "dt_sin_pri": "2025-03-10",
      "sem_pri": "202511",
      "nu_idade_n": "4030",
      "cs_sexo": "F",
      "cs_gestant": "5",
      "cs_raca": "1",
      "cs_escol_n": "8",
      "sg_uf": "35",
      "id_mn_resi": "354990",
      "id_rg_resi": "1351",
      "id_pais": "1",
      "nduplic_n": "nan",
      "dt_digita": "2025-03-11",
      "cs_flxret": "0",
      "flxrecebi": "nan",
      "migrado_w": "nan",
      "dt_invest": "2025-03-11",
      "id_ocupa_n": "231340",
      "dt_soro": "nan",
      "resul_soro": "nan",
      "histopa_n": "nan",
      "dt_viral": "nan",
      "resul_vi_n": "nan",
      "sorotipo": "nan",
      "imunoh_n": "nan",
      "dt_pcr": "nan",
      "resul_pcr_": "nan",
      "classi_fin": "

In [15]:
print(
    json.dumps(
        dados,
        indent=2,
        ensure_ascii=False
    )[:5000]
)

{
  "dengue": [
    {
      "tp_not": "2",
      "id_agravo": "A90",
      "dt_notific": "2025-03-11",
      "sem_not": "202511",
      "nu_ano": "2025",
      "sg_uf_not": "35",
      "id_municip": "354990",
      "id_regiona": "1351",
      "id_unidade": "3708616",
      "dt_sin_pri": "2025-03-10",
      "sem_pri": "202511",
      "nu_idade_n": "4030",
      "cs_sexo": "F",
      "cs_gestant": "5",
      "cs_raca": "1",
      "cs_escol_n": "8",
      "sg_uf": "35",
      "id_mn_resi": "354990",
      "id_rg_resi": "1351",
      "id_pais": "1",
      "nduplic_n": "nan",
      "dt_digita": "2025-03-11",
      "cs_flxret": "0",
      "flxrecebi": "nan",
      "migrado_w": "nan",
      "dt_invest": "2025-03-11",
      "id_ocupa_n": "231340",
      "dt_soro": "nan",
      "resul_soro": "nan",
      "histopa_n": "nan",
      "dt_viral": "nan",
      "resul_vi_n": "nan",
      "sorotipo": "nan",
      "imunoh_n": "nan",
      "dt_pcr": "nan",
      "resul_pcr_": "nan",
      "classi_fin": "